# HTR with Claude

Requires an API key from platform.claude.ai

In [65]:
import anthropic
from datetime import datetime
from dotenv import load_dotenv
import json
import os
from pathlib import Path
import regex

In [2]:
base_directory = "../memories_crawl/scans/bhic/Eindhoven/deel_84"

## 1. Find act-initial text block with Claude

Processing a single image costs about 0.5 cents

In [3]:
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [4]:
prompt = """Dear Claude, here is an image displaying two pages with Dutch text 
related to an inheritance. I am interested in the information in the text block 
on the top right of the right page. Could check if that part contains a text like: 
"Memorie van aangifte der nalatenschap van"? If that is the case, can you give me 
the information which follows next? This is 1. the name of the deceased, 2. the 
place of death, and 3. the date of death. Both place and date could be missing. 
If you see a big number next to the text block, that is the act number, which is 
interesting as well. Please return this all information in well-formatted JSON 
format with the keys "act", "name", "place" and "date" and without any comments. 
If the top right of the right page contains a different text or no text at all, 
please return an empty JSON structure."""

In [5]:
starting_pages = [x for x in range(1, 101)]
results = []
for file_name in sorted(os.listdir(base_directory)):
    try:
        page_number = int(regex.sub("^0+", "", file_name.split("_")[-1].split('.')[0]))
    except ValueError:
        continue
    if page_number in starting_pages:
        try:
            upload_response = client.beta.files.upload(file=Path(os.path.join(base_directory, file_name)))
            message = client.beta.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1024,
                messages=[
                     {"role": "user", 
                      "content": [
                        {"type": "image",
                         "source": {"type": "file",
                                    "file_id": upload_response.id
                                   }
                        },
                        {"type": "text", "text": prompt}
                     ]}
                ],
                betas=["files-api-2025-04-14"]
            )
            results.append(message.content[0].text)
        finally:
            client.beta.files.delete(upload_response.id)

KeyboardInterrupt: 

In [59]:
def str2dict(string):
    groups = regex.search(r"^(.*)```json(.*)```(.*)$", string.strip(), flags=regex.DOTALL)
    person_dict = json.loads(groups.group(2))
    comment_prefix = groups.group(1).strip()
    comment_suffix = groups.group(1).strip()
    if comment_prefix:
        if comment_suffix:
            person_dict["comment"] = " ".join([comment_prefix, comment_suffix])
        else:
            person_dict["comment"] = comment_prefix
    elif comment_suffix:
        person_dict["comment"] = comment_suffix
    return person_dict

In [71]:
def save_json(results_json):
    today = datetime.strftime(datetime.now(), "%Y%m%d")
    with open(f"output_{today}.json", "w") as f:
        json.dump(results_json, f)
        f.close()

In [72]:
results_json = []
for result in results:
    results_json.append(str2dict(result))
save_json(results_json)

## 2. Link HTR information to metadata

In [73]:
with open(os.path.join(base_directory, "deeds.json"), "r") as f:
    data = json.load(f)
    f.close()

In [78]:
target_name = results_json[1]["name"]

In [79]:
for act in data:
    # if naam_volledig in person and person.naam_volledig == target_name:
    if "personen" in act:
        for person in act["personen"]:
            if "naam_volledig" in person and person["naam_volledig"] == target_name:
                print(person)

{'person_id': '5a2a68b6-464f-11e3-a747-d206bceb4d38', 'voornaam': 'Dorothea', 'tussenvoegsel': 'van', 'geslachtsnaam': 'Ummelen', 'naam_volledig': 'Dorothea van Ummelen', 'geslacht': '', 'datum_overlijden': '1889-01-10', 'plaats_overlijden': 'Aalst', 'rol': 'overledene'}
